In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import seaborn as sns
from Bio import Entrez, Medline
import mermaid
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript
import subprocess, sys, os, ssl, certifi
import re
from pathlib import Path

In [8]:
import os 
from pathlib import Path
root = os.getcwd()
folders = {
    "systematic_review": f"{root}/systematic_review",
        "protocol": f"{root}/systematic_review/protocol",
            "prospero": f"{root}/systematic_review/protocol/prospero",
            "cochrane": f"{root}/systematic_review/protocol/cochrane",
        "search_strategy": f"{root}/systematic_review/search_strategy",
        "search": f"{root}/systematic_review/search",
        "deduplication": f"{root}/systematic_review/deduplication",
        "screening": f"{root}/systematic_review/screening",
            "title_abstract": f"{root}/systematic_review/screening/title_abstract_screening", 
            "pdf": f"{root}/systematic_review/screening/PDF",
            "full_text": f"{root}/systematic_review/screening/full_text_screening", 
    "data_collection": f"{root}/data_collection",
        "database": f"{root}/data_collection/database",
    "meta-analysis": f"{root}/meta-analysis",
    "manuscript": f"{root}/manuscript"
}

print(folders)
for x, y in folders.items():
    filename = f"{x}"
    path = Path(f"{y}")
    os.makedirs(path, exist_ok = True)
    globals()[filename] = path

{'systematic_review': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review', 'protocol': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/protocol', 'prospero': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/protocol/prospero', 'cochrane': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/protocol/cochrane', 'search_strategy': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/search_strategy', 'search': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/search', 'deduplication': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/deduplication', 'screening': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/screening', 'title_abstract': 'G:\\My Drive\\network_meta-analysis\\app\\voila\\render/systematic_review/screening/title_abstract_screening', 'pdf': 'G:\\My Drive\\network_meta-analysis\\

In [12]:
ssl._create_default_https_context = lambda: ssl.create_default_context(
    cafile=certifi.where()
)
import os
# create search strategy using structured inputs

question = input("Do you already have a search strategy file saved?") # get rid of this
filename = input("Enter the file name of the search strategy: ") # use file upload widget ! and present the results as data table widgets !!! 
file = f"{search_strategy}/{filename}.txt"

if question == "no":
    parts = []
    string = []
    while True:
        term = input("Enter the search string: ")
        field = input("Enter the field type: ")
        string.append(f"'{term}'[{field}]")
        boolean = input("Enter the Boolean operator (e.g., OR, AND, NOT): ")
        
        if boolean == "OR":
            continue
    
        parts.append("(" + " OR ".join(string) + ")")
        string = []
    
        if boolean == "":
            break
            
        parts.append(boolean)
        
    query = " ".join(parts) # query = search strategy FROM HERE
    with open(file, "w") as f:
        f.write(query)
    
with open(file, "r") as f:
    query = f.read()

query = f"{query}"

# use NCBI's e-utitilies to pull PMIDs using e-search.

Entrez.email = "dkim246@jhmi.edu"
Entrez.api_key = 'bb1c481d8e167acd16f3616593c18b3aab08'

handle = Entrez.esearch(db= "pubmed", 
                        term = query, 
                        usehistory = "y", 
                        retmax = 2000,
                        retmode = "xml")

pmid = Entrez.read(handle)

pmid = pmid['IdList']
pmid = ",".join(pmid) # list to string
#with open(f"./data/pmid_{filename}.txt", 'w') as f:
#    f.write(pmid)
os.makedirs(f"{search}/pubmed/pmid/", exist_ok = True)
with open(f"{search}/pubmed/pmid/{filename}.txt", 'w') as f:
    f.write(pmid)
handle.close()

# ncbi e-summary
handle = Entrez.esummary(db= "pubmed", 
                         id = pmid, 
                         retmode = "xml", 
                         usehistory = "y", 
                         retmax = 2000)

xml = handle.read()
#xml_file = f"./data/{filename}.xml"
os.makedirs(f"{search}/pubmed/xml/", exist_ok = True)
xml_file = f"{search}/pubmed/xml/{filename}.xml"
with open(xml_file, "wb") as f:
    f.write(xml)   
handle.close()

import xml.etree.ElementTree as ET

tree = ET.parse(f"{xml_file}")
root = tree.getroot()

docsum = root[0]

def xml_parse(docsum):
    df = {}
    df["pmid"] = docsum.find("Id").text
    for item in docsum.findall("Item"):
        key = item.attrib.get("Name")
        if item.attrib.get("Type") == "List":
            values = [sub.text for sub in item.findall("Item") if sub.text]
            df[key] = values
        else:
            df[key] = item.text
    return df
records = [xml_parse(doc) for doc in root.findall(".//DocSum")]
df = pd.DataFrame(records)

os.makedirs(f"{search}/pubmed/", exist_ok = True)
csv_file = f"{search}/pubmed/{filename}.csv"
df.to_csv(csv_file, encoding = "utf-8")

# using e-fetch, the abstracts are pulled

handle = Entrez.efetch(
    db="pubmed",
    id=pmid,
    rettype="medline",
    retmode="text"
)

text = list(Medline.parse(handle))
data = pd.DataFrame(text)
data_csv = data.map(lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x)
os.makedirs(f"{search}/pubmed/medline/", exist_ok = True)
data_csv.to_csv(f"{search}/pubmed/medline/{filename.replace("pm","md")}.csv", index=False)
globals()[f"{filename.replace("pm","md")}"] = data
handle.close()

abstracts = pd.DataFrame(text)[["PMID", "AB"]]
abstracts.rename(columns = {"PMID":"pmid", "AB":"abstract"}, inplace = True)
df = df.merge(abstracts, on = "pmid", how = "left")
df['year'] = df['PubDate'].str[:4]
csv_file = f"{search}/pubmed/{filename}.csv"
df.to_csv(csv_file, encoding = "utf-8")
num = len(df)
print(f"Number of records found: {num}")
data.info()



Do you already have a search strategy file saved? no
Enter the file name of the search strategy:  anterior cruciate ligament
Enter the search string:  anterior cruciate ligament
Enter the field type:  tiab
Enter the Boolean operator (e.g., OR, AND, NOT):  


URLError: <urlopen error [Errno 11001] getaddrinfo failed>